# Smart Retention Engine
## Notebook 07: From Prediction → Prescription → Revenue Impact

This notebook goes **beyond churn prediction** to answer the question stakeholders actually care about:

> *"We know customers are churning. What should we DO about each one, and what's the ROI?"*

**Three components:**
1. **Customer Health Score (0-100)** — Composite metric combining 5 weighted signals
2. **Churn Driver Identification** — WHY each customer might leave (7 driver categories)
3. **Personalized Retention Strategy** — WHAT to do, with cost-benefit ROI analysis

**Data Source**: Real IBM Telco Customer Churn dataset from Kaggle (7,043 customers)

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from retention_engine import (
    CustomerHealthScorer, ChurnDriverAnalyzer, 
    RetentionStrategyEngine, run_full_retention_analysis
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print('Libraries imported successfully!')

## 1. Run Full Retention Engine Pipeline

In [ ]:
# Run the complete retention analysis pipeline
df, summary, model_metrics = run_full_retention_analysis('../data/raw/master_dataset.csv')

## 2. Customer Health Score Analysis

The Health Score is a composite metric (0-100) combining:
- **Satisfaction** (30%) — Direct satisfaction signal
- **Usage Engagement** (25%) — Are they actively using the service?
- **Payment Reliability** (20%) — On-time payment history
- **Tenure Loyalty** (15%) — How long they've been with us
- **Digital Engagement** (10%) — Web/app session activity

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Customer Health Score Analysis', fontsize=18, fontweight='bold')

# 1. Health Score Distribution
axes[0, 0].hist(df['customer_health_score'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0, 0].axvline(df['customer_health_score'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {df["customer_health_score"].mean():.1f}')
axes[0, 0].set_title('Health Score Distribution')
axes[0, 0].set_xlabel('Health Score (0-100)')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].legend()

# 2. Health Category Breakdown
health_counts = df['health_category'].value_counts()
colors = ['#d32f2f', '#ff9800', '#4caf50', '#2196f3']
axes[0, 1].pie(health_counts.values, labels=health_counts.index, autopct='%1.1f%%',
              colors=colors, startangle=90)
axes[0, 1].set_title('Health Category Distribution')

# 3. Health Score vs Churn Rate
df['health_bin'] = pd.cut(df['customer_health_score'], bins=10)
churn_by_health = df.groupby('health_bin')['churned'].mean() * 100
axes[1, 0].bar(range(len(churn_by_health)), churn_by_health.values, color='coral')
axes[1, 0].set_title('Churn Rate by Health Score Range')
axes[1, 0].set_xlabel('Health Score Range (Low → High)')
axes[1, 0].set_ylabel('Churn Rate (%)')
axes[1, 0].set_xticks(range(len(churn_by_health)))
axes[1, 0].set_xticklabels([str(x) for x in churn_by_health.index], rotation=45, ha='right', fontsize=7)

# 4. Health Score vs ARPU (segment view)
for cat in ['Critical', 'At Risk', 'Stable', 'Healthy']:
    subset = df[df['health_category'] == cat]
    if len(subset) > 0:
        axes[1, 1].scatter(subset['customer_health_score'], subset['arpu'], 
                          alpha=0.3, label=f'{cat} ({len(subset):,})', s=10)
axes[1, 1].set_title('Health Score vs ARPU by Segment')
axes[1, 1].set_xlabel('Health Score')
axes[1, 1].set_ylabel('ARPU ($)')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Churn Driver Analysis

Instead of just predicting WHO will churn, we identify WHY — which enables targeted interventions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Driver Distribution
driver_counts = df['churn_driver_label'].value_counts()
colors_d = plt.cm.Set2(np.linspace(0, 1, len(driver_counts)))
axes[0].barh(driver_counts.index, driver_counts.values, color=colors_d)
axes[0].set_title('Primary Churn Driver Distribution', fontsize=14)
axes[0].set_xlabel('Number of Customers')
for i, v in enumerate(driver_counts.values):
    axes[0].text(v + 30, i, f'{v:,} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)

# 2. Driver vs Actual Churn Rate
driver_churn = df.groupby('churn_driver_label')['churned'].mean().sort_values(ascending=True) * 100
axes[1].barh(driver_churn.index, driver_churn.values, color='salmon')
axes[1].set_title('Actual Churn Rate by Driver Category', fontsize=14)
axes[1].set_xlabel('Churn Rate (%)')
for i, v in enumerate(driver_churn.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Driver summary table
driver_summary = df.groupby('churn_driver_label').agg({
    'customer_id': 'count',
    'churned': 'mean',
    'arpu': 'mean',
    'customer_health_score': 'mean',
    'satisfaction_score': 'mean'
}).round(2)
driver_summary.columns = ['Customers', 'Churn Rate', 'Avg ARPU', 'Avg Health', 'Avg Satisfaction']
driver_summary['Churn Rate'] = (driver_summary['Churn Rate'] * 100).round(1).astype(str) + '%'
driver_summary = driver_summary.sort_values('Customers', ascending=False)
print('\nChurn Driver Summary Table:')
driver_summary

## 4. Retention Strategy Recommendations

For each at-risk customer, the engine recommends a specific action with **cost-benefit ROI analysis**.

Key insight: We prioritize by **expected value** (revenue at risk × churn probability), not just churn probability. A high-ARPU customer with moderate risk deserves more attention than a low-ARPU customer with high risk.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Retention Strategy Analysis', fontsize=18, fontweight='bold')

# 1. Strategy Distribution
strat_counts = df['strategy_name'].value_counts()
axes[0, 0].barh(range(len(strat_counts)), strat_counts.values, color=plt.cm.Paired(np.linspace(0, 1, len(strat_counts))))
axes[0, 0].set_yticks(range(len(strat_counts)))
axes[0, 0].set_yticklabels(strat_counts.index, fontsize=8)
axes[0, 0].set_title('Strategy Distribution')
axes[0, 0].set_xlabel('Number of Customers')

# 2. ROI by Strategy
at_risk = df[df['strategy_key'] != 'no_action']
roi_by_strat = at_risk.groupby('strategy_name').agg({
    'intervention_cost': 'sum',
    'expected_revenue_saved': 'sum'
})
roi_by_strat['ROI'] = (roi_by_strat['expected_revenue_saved'] - roi_by_strat['intervention_cost']) / roi_by_strat['intervention_cost']
roi_by_strat = roi_by_strat.sort_values('ROI', ascending=True)
axes[0, 1].barh(range(len(roi_by_strat)), roi_by_strat['ROI'].values, color='seagreen')
axes[0, 1].set_yticks(range(len(roi_by_strat)))
axes[0, 1].set_yticklabels(roi_by_strat.index, fontsize=8)
axes[0, 1].set_title('ROI by Retention Strategy')
axes[0, 1].set_xlabel('Return on Investment (x)')
for i, v in enumerate(roi_by_strat['ROI'].values):
    axes[0, 1].text(v + 0.1, i, f'{v:.1f}x', va='center', fontsize=9)

# 3. Budget Allocation
budget = at_risk.groupby('strategy_name')['intervention_cost'].sum().sort_values(ascending=False)
axes[1, 0].pie(budget.values, labels=[s[:25] for s in budget.index], autopct='%1.1f%%',
              colors=plt.cm.Set3(np.linspace(0, 1, len(budget))))
axes[1, 0].set_title(f'Budget Allocation (Total: ${budget.sum():,.0f})')

# 4. Priority Score vs Expected Revenue Saved
top_100 = at_risk.nsmallest(100, 'retention_priority_rank')
scatter = axes[1, 1].scatter(top_100['ml_churn_probability'], top_100['expected_revenue_saved'],
                            c=top_100['arpu'], cmap='YlOrRd', s=50, alpha=0.7)
plt.colorbar(scatter, ax=axes[1, 1], label='ARPU ($)')
axes[1, 1].set_title('Top 100 Priority Customers')
axes[1, 1].set_xlabel('ML Churn Probability')
axes[1, 1].set_ylabel('Expected Revenue Saved ($)')

plt.tight_layout()
plt.show()

## 5. Churn Prediction Model Performance

In [ ]:
# Model metrics
print('=== CHURN PREDICTION MODEL PERFORMANCE ===')
print(f'AUC-ROC: {model_metrics["auc_roc"]:.4f}')
print(f'CV AUC:  {model_metrics["cv_auc_mean"]:.4f} ± {model_metrics["cv_auc_std"]:.4f}')

print('\nClassification Report:')
report = model_metrics['classification_report']
for label in ['0', '1']:
    r = report[label]
    label_name = 'Retained' if label == '0' else 'Churned'
    print(f'  {label_name}: P={r["precision"]:.3f} R={r["recall"]:.3f} F1={r["f1-score"]:.3f}')

# Feature importance
fi = pd.Series(model_metrics['feature_importance']).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 8))
fi.tail(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Feature Importances (Gradient Boosting)', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Executive Summary & Business Impact

In [ ]:
print('=' * 60)
print('EXECUTIVE SUMMARY: SMART RETENTION ENGINE')
print('=' * 60)

print(f'\n📊 CUSTOMER BASE OVERVIEW')
print(f'  Total customers:    {summary["total_customers"]:,}')
print(f'  At-risk customers:  {summary["at_risk_customers"]:,} ({summary["at_risk_pct"]:.1%})')
print(f'  Monthly revenue:    ${summary["total_monthly_revenue"]:,.2f}')

print(f'\n💰 FINANCIAL IMPACT')
print(f'  Revenue at risk:    ${summary["at_risk_monthly_revenue"]:,.2f}/month')
print(f'  Intervention cost:  ${summary["total_intervention_cost"]:,.2f} (one-time)')
print(f'  Expected savings:   ${summary["total_expected_revenue_saved"]:,.2f}/year') 
print(f'  Overall ROI:        {summary["overall_roi"]:.1f}x')
print(f'  Net benefit:        ${summary["total_expected_revenue_saved"] - summary["total_intervention_cost"]:,.2f}')

print(f'\n🎯 TOP 10 PRIORITY CUSTOMERS')
top10 = pd.DataFrame(summary['top_priority_customers'])
top10_display = top10[['customer_id', 'arpu', 'ml_churn_probability', 
                       'customer_health_score', 'churn_driver_label', 
                       'strategy_name', 'expected_revenue_saved']].copy()
top10_display.columns = ['ID', 'ARPU', 'Churn Prob', 'Health', 'Driver', 'Strategy', 'Rev Saved']
top10_display['ID'] = top10_display['ID'].str[:8] + '...'
top10_display['ARPU'] = top10_display['ARPU'].apply(lambda x: f'${x:.2f}')
top10_display['Churn Prob'] = top10_display['Churn Prob'].apply(lambda x: f'{x:.1%}')
top10_display['Rev Saved'] = top10_display['Rev Saved'].apply(lambda x: f'${x:.0f}')
print(top10_display.to_string(index=False))

print(f'\n🤖 MODEL PERFORMANCE')
print(f'  AUC-ROC:            {summary["model_auc"]:.4f}')
print(f'  Cross-Validated:    {summary["model_cv_auc"]:.4f}')

print(f'\n✅ RECOMMENDATION')
print(f'  Deploy the retention engine to automatically flag at-risk customers')
print(f'  and assign interventions. Expected annual revenue protection:')
print(f'  ${summary["total_expected_revenue_saved"]:,.2f} for a ${summary["total_intervention_cost"]:,.2f} investment.')
print('=' * 60)

In [ ]:
# Save enriched dataset with health scores and retention recommendations
import os
os.makedirs('../data/processed', exist_ok=True)

output_cols = [
    'customer_id', 'age', 'gender', 'city', 'plan_type', 'tenure_months',
    'arpu', 'churned', 'customer_health_score', 'health_category',
    'churn_driver_label', 'churn_driver_score', 'ml_churn_probability',
    'strategy_name', 'intervention_cost', 'expected_revenue_saved',
    'expected_roi', 'retention_priority_rank'
]
available_cols = [c for c in output_cols if c in df.columns]
df[available_cols].to_csv('../data/processed/retention_analysis.csv', index=False)
print(f'Saved retention analysis to data/processed/retention_analysis.csv ({len(df):,} rows)')